# Installing interp-modified boltz

In [ ]:
!rm -rf "boltz"
!git clone "https://github.com/DanClark1/boltz.git"
!cp -r "boltz" "boltz_repo"
!pip install -e boltz_repo
!rm -rf "boltz"
!cp -r "boltz_repo/src/boltz" "./boltz"
!rm -rf "boltz_repo"

# Collecting activations

In [ ]:
import os, re, torch
import boltz
from boltz.analysis.interp import register_metadata_hook, decode_res_types, get_recycling_step

DRIVE    = "/content/drive/MyDrive"
YAML_DIR = f"{DRIVE}/gdl/examples"
OUT_DIR  = f"{DRIVE}/final_experiments"
os.makedirs(OUT_DIR, exist_ok=True)
EXAMPLES = ["prot_no_msa", "prot", "multimer", "cyclic_prot", "ligand"]

model = boltz.load_model("boltz1", device="cuda", use_kernels=False)

for example in EXAMPLES:
    save_base = f"{OUT_DIR}/{example}/raw"
    os.makedirs(save_base, exist_ok=True)
    acts_path = f"{save_base}/activations.pt"
    if os.path.exists(acts_path):
        print(f"[{example}] Already captured – skipping inference.")
        continue

    print(f"\n{'='*60}\n[{example}] Running inference …")
    metadata, meta_handle = register_metadata_hook(model)

    activations = {}

    def get_hook(layer_name, component_name):
        def hook(module, input, output):
            # Overwrite each step → only last recycling step survives
            activations.setdefault(layer_name, {})[component_name] = \
                output.detach().cpu().clone()
        return hook

    hook_handles = []
    for name, module in model.named_modules():
        if "AttentionPairBias" in str(type(module)):
            hook_handles += [
                module.proj_q.register_forward_hook(get_hook(name, "q")),
                module.proj_k.register_forward_hook(get_hook(name, "k")),
                module.proj_z.register_forward_hook(get_hook(name, "bias")),
                module.proj_o.register_forward_hook(get_hook(name, "o")),
            ]

    yaml_path = f"{YAML_DIR}/{example}.yaml"
    results   = boltz.predict(model, yaml_path, use_msa_server=True,
                              recycling_steps=3, diffusion_samples=1)

    res_names = decode_res_types(metadata["res_type"],
                                  metadata.get("token_pad_mask"))
    print(f"  Sequence length: {len(res_names)} tokens")

    # Wrap single tensors in lists so get_recycling_step(…, step=-1) still works
    acts_to_save = {
        layer: {comp: [tensor] for comp, tensor in comps.items()}
        for layer, comps in activations.items()
    }

    torch.save(acts_to_save,         f"{save_base}/activations.pt")
    torch.save(results[0]["coords"], f"{save_base}/coords.pt")
    torch.save({"res_names": res_names}, f"{save_base}/meta.pt")

    for h in hook_handles: h.remove()
    meta_handle.remove()
    activations.clear()
    print(f"  Saved to {save_base}")

print("\nInference complete.")

# Processing activations + generating plots

In [ ]:
import os, re, numpy as np, torch, matplotlib.pyplot as plt
import gc
from boltz.analysis.interp import (
    get_recycling_step,
    compute_ca_coords,
    compute_distance_matrix,
    run_kl_analysis,
    plot_kl_heatmaps_separate,
    plot_kl_heatmap_combined,
    plot_semantic_peaks,
    plot_geometric_peaks,
    plot_geo_head_correlation_heatmap,
    plot_diagonal_head_correlation_heatmap,
    plot_bias_correlation_gallery,
    plot_structure_vs_top_geo_bias,
    plot_bias_sampled_layers,
    plot_toeplitz_heatmap,
    plot_toeplitz_gallery,
    compute_seq_window_mass,
    plot_seq_window_mass_heatmaps,
    plot_seq_window_mass_gallery,
)

DRIVE      = "/content/drive/MyDrive"
OUT_ROOT   = f"{DRIVE}/final_experiments"
GLOBAL_DIR = f"{OUT_ROOT}/combined"
EXAMPLES   = ["prot_no_msa", "prot", "multimer", "cyclic_prot", "ligand"]
# EXAMPLES = ["ligand"]
os.makedirs(GLOBAL_DIR, exist_ok=True)

EXAMPLE_COLORS = {
    "prot_no_msa": "#5B8DB8",
    "prot":        "#E07B7B",
    "multimer":    "#5BA85B",
    "cyclic_prot": "#C05A2A",
}



EXAMPLE_LABELS = {
    "prot_no_msa": "α3D (Single Seq)",
    "prot":        "α3D (with MSA)",
    "multimer":    "Heterodimer ",
    "cyclic_prot": "Cyclic Peptide",
    "ligand":      "Protein-Ligand"
}

FIG_SIZES = {
    "rq1_heatmap":        None,
    "rq1_combined":       None,
    "rq1_sem_peaks_line": (5, 5),
    "rq1_sem_peaks_bars": None,
    "rq1_geo_peaks_line": (5, 5),
    "rq1_geo_peaks_bars": None,
    "rq1_corr":           None,
    "rq1_diag_corr":      None,
    "rq1_corr_gallery":   None,
    "rq1_toeplitz":       None,
    "rq1_seq_mass":        None,
    "rq2_struct":         None,
    "rq2_sampled":        None,
    "overlay":            (14, 8),
}

GEO_THRESHOLD = 0.5

def _layer_idx(name):
    m = re.search(r"layers\.(\d+)", name)
    return int(m.group(1)) if m else -1


def process_example(example):
    """Run full analysis for one protein; return only the lightweight overlay lines."""
    raw_dir = f"{OUT_ROOT}/{example}/raw"
    out_dir = f"{OUT_ROOT}/{example}"
    os.makedirs(out_dir, exist_ok=True)

    acts_path = f"{raw_dir}/activations.pt"
    if not os.path.exists(acts_path):
        print(f"[{example}] No activations found – run Cell 10 first.")
        return None

    label = EXAMPLE_LABELS.get(example, example)

    # Helper: build a save path with the structure name embedded in the filename
    def sp(stem):
        return f"{out_dir}/{example}_{stem}.png"

    print(f"\n{'='*60}\n[{example}] Loading data …")
    all_acts  = torch.load(acts_path,               map_location="cpu")
    coords    = torch.load(f"{raw_dir}/coords.pt",  map_location="cpu")
    meta      = torch.load(f"{raw_dir}/meta.pt",    map_location="cpu")
    res_names = meta["res_names"]

    layer_names  = sorted(
        [k for k in all_acts if k.startswith("pairformer_module.")],
        key=_layer_idx,
    )
    layer_labels = [_layer_idx(n) for n in layer_names]
    print(f"  Layers: {len(layer_names)}   Residues: {len(res_names)}")

    acts      = get_recycling_step(all_acts, step=-1)
    del all_acts

    ca_coords = compute_ca_coords(coords, res_names, sample_idx=0)
    ca_dist   = compute_distance_matrix(ca_coords)
    del coords, ca_coords

    # ── KL analysis ───────────────────────────────────────────────────
    print("  Running KL analysis …", flush=True)
    geo_raw, sem_raw = run_kl_analysis(acts, layer_names, num_trials=5)

    total     = geo_raw + sem_raw + 1e-10
    geo_ratio = geo_raw / total
    sem_ratio = sem_raw / total

    geo_p95  = np.percentile(geo_raw, 95)
    sem_p95  = np.percentile(sem_raw, 95)
    geo_norm = np.clip(geo_raw / (geo_p95 + 1e-10), 0, 1)
    sem_norm = np.clip(sem_raw / (sem_p95 + 1e-10), 0, 1)

    print(f"  ratio  geo={geo_ratio.mean():.3f}  sem={sem_ratio.mean():.3f}")
    print(f"  raw    geo={geo_norm.mean():.3f}  sem={sem_norm.mean():.3f}")

    # ── RQ1-1  Heatmaps – ratio & raw ────────────────────────────────
    for tag, geo_s, sem_s in [("ratio", geo_ratio, sem_ratio),
                               ("raw",   geo_norm,  sem_norm)]:
        plot_kl_heatmaps_separate(
            geo_s, sem_s, layer_labels,
            save_geo=sp(f"rq1_{tag}_geo_heatmap"),
            save_sem=sp(f"rq1_{tag}_sem_heatmap"),
            figsize=FIG_SIZES["rq1_heatmap"],
            title_geo=f"Geometric importance ({tag})  —  {label}",
            title_sem=f"Semantic importance ({tag})  —  {label}",
        )
        plot_kl_heatmap_combined(
            geo_s, sem_s, layer_labels,
            save_path=sp(f"rq1_{tag}_combined_heatmap"),
            figsize=FIG_SIZES["rq1_combined"],
            suptitle=f"Geometric vs Semantic importance ({tag})  —  {label}",
        )

    # ── RQ1-3  Semantic peaks – ratio & raw ──────────────────────────
    _, _, sem_line_ratio, _ = plot_semantic_peaks(
        acts, layer_names, res_names, sem_ratio, layer_labels,
        n_peaks=3, sem_threshold=0.4,
        save_path_line=sp("rq1_ratio_sem_peaks_line"),
        save_path_bars=sp("rq1_ratio_sem_peaks_bars"),
        figsize_line=FIG_SIZES["rq1_sem_peaks_line"],
        figsize_bars=FIG_SIZES["rq1_sem_peaks_bars"],
        title_line=f"Semantic routing across layers (ratio)  —  {label}",
    )
    _, _, sem_line_raw, _ = plot_semantic_peaks(
        acts, layer_names, res_names, sem_norm, layer_labels,
        n_peaks=3, sem_threshold=0.4,
        save_path_line=sp("rq1_raw_sem_peaks_line"),
        save_path_bars=sp("rq1_raw_sem_peaks_bars"),
        figsize_line=FIG_SIZES["rq1_sem_peaks_line"],
        figsize_bars=FIG_SIZES["rq1_sem_peaks_bars"],
        title_line=f"Semantic routing across layers (raw norm.)  —  {label}",
    )

    # ── RQ1-4  Geometric peaks – ratio & raw ─────────────────────────
    _, _, geo_line_ratio, _ = plot_geometric_peaks(
        acts, layer_names, res_names, geo_ratio, layer_labels,
        n_peaks=3, geo_threshold=0.4,
        save_path_line=sp("rq1_ratio_geo_peaks_line"),
        save_path_bars=sp("rq1_ratio_geo_peaks_bars"),
        figsize_line=FIG_SIZES["rq1_geo_peaks_line"],
        figsize_bars=FIG_SIZES["rq1_geo_peaks_bars"],
        title_line=f"Geometric routing across layers (ratio)  —  {label}",
    )
    _, _, geo_line_raw, _ = plot_geometric_peaks(
        acts, layer_names, res_names, geo_norm, layer_labels,
        n_peaks=3, geo_threshold=0.4,
        save_path_line=sp("rq1_raw_geo_peaks_line"),
        save_path_bars=sp("rq1_raw_geo_peaks_bars"),
        figsize_line=FIG_SIZES["rq1_geo_peaks_line"],
        figsize_bars=FIG_SIZES["rq1_geo_peaks_bars"],
        title_line=f"Geometric routing across layers (raw norm.)  —  {label}",
    )

    # ── RQ1-5  Bias–structure correlation heatmap ─────────────────────
    _, corr_mat = plot_geo_head_correlation_heatmap(
        acts, layer_names, ca_dist, res_names, geo_ratio,
        geo_threshold=GEO_THRESHOLD,
        save_path=sp("rq1_geo_corr_heatmap"),
        figsize=FIG_SIZES["rq1_corr"],
        title=f"Bias–structure correlation (geo heads >{GEO_THRESHOLD:.0%})  —  {label}",
    )

    # ── RQ1-5b Bias–diagonal (sequential) correlation heatmap ─────────
    # All heads (no threshold)
    _, diag_corr_mat_all = plot_diagonal_head_correlation_heatmap(
        acts, layer_names, res_names,
        save_path=sp("rq1_diag_corr_heatmap_all"),
        figsize=FIG_SIZES["rq1_diag_corr"],
        title=f"Bias–diagonal correlation (all heads)  —  {label}",
    )
    # Geo-thresholded heads only
    _, diag_corr_mat = plot_diagonal_head_correlation_heatmap(
        acts, layer_names, res_names, geo_ratio,
        geo_threshold=GEO_THRESHOLD,
        save_path=sp("rq1_diag_corr_heatmap_geo"),
        figsize=FIG_SIZES["rq1_diag_corr"],
        title=f"Bias–diagonal correlation (geo heads >{GEO_THRESHOLD:.0%})  —  {label}",
    )

    # ── RQ1-5c  Toeplitz decomposition heatmap ────────────────────────
    _, toeplitz_mat = plot_toeplitz_heatmap(
        acts, layer_names, res_names,
        save_path=sp("rq1_toeplitz_heatmap_all"),
        figsize=FIG_SIZES["rq1_toeplitz"],
        title=f"Toeplitz η² (all heads)  —  {label}",
    )
    _, toeplitz_mat_geo = plot_toeplitz_heatmap(
        acts, layer_names, res_names, geo_ratio,
        geo_threshold=GEO_THRESHOLD,
        save_path=sp("rq1_toeplitz_heatmap_geo"),
        figsize=FIG_SIZES["rq1_toeplitz"],
        title=f"Toeplitz η² (geo heads >{GEO_THRESHOLD:.0%})  —  {label}",
    )

    # ── RQ1-5d  Toeplitz gallery ──────────────────────────────────────
    if (~np.isnan(toeplitz_mat)).sum() >= 3:
        plot_toeplitz_gallery(
            acts, layer_names, res_names, toeplitz_mat,
            zoom=min(80, len(res_names)),
            save_path=sp("rq1_toeplitz_gallery"),
            title=f"Toeplitz η²  —  {label}",
        )

    # ── RQ1-5e  Attention mass within sequence windows ───────────────
    SEQ_WINDOWS = (5, 15, 30)
    seq_mass = compute_seq_window_mass(
        acts, layer_names, res_names, windows=SEQ_WINDOWS,
    )
    plot_seq_window_mass_heatmaps(
        seq_mass, layer_names,
        save_path=sp("rq1_seq_mass_heatmaps"),
        figsize=FIG_SIZES["rq1_seq_mass"],
        title_prefix=label,
    )
    if (~np.isnan(seq_mass[15])).sum() >= 3:
        plot_seq_window_mass_gallery(
            acts, layer_names, res_names, seq_mass,
            window=15,
            zoom=min(80, len(res_names)),
            save_path=sp("rq1_seq_mass_gallery"),
            title=f"Attention mass within \u00b115 positions  —  {label}",
        )

    # ── RQ1-6  Bias correlation gallery ───────────────────────────────
    if corr_mat is not None and (~np.isnan(corr_mat)).sum() >= 3:
        prox_mat = 1.0 / (ca_dist + 1.0)
        plot_bias_correlation_gallery(
            acts, layer_names, res_names, corr_mat,
            prox_mat=prox_mat,
            geo_threshold=GEO_THRESHOLD,
            zoom=min(80, len(res_names)),
            save_path=sp("rq1_corr_gallery"),
            figsize=FIG_SIZES["rq1_corr_gallery"],
            title=f"Bias–structure Spearman r  —  {label}",
            contact_title=f"Ground-truth Cα proximity  —  {label}",
        )

    # ── RQ2-1  Structure proximity vs bias attention ───────────────────
    plot_structure_vs_top_geo_bias(
        acts, layer_names, ca_dist, res_names, geo_ratio,
        zoom=min(80, len(res_names)),
        save_path=sp("rq2_structure_vs_bias_combined"),
        save_path_prox=sp("rq2_structure_proximity"),
        save_path_bias=sp("rq2_geo_bias_attention"),
        figsize=FIG_SIZES["rq2_struct"],
        suptitle=f"Structure proximity vs geometric attention  —  {label}",
    )

    # ── RQ2-2  Sampled bias matrices ──────────────────────────────────
    plot_bias_sampled_layers(
        acts, layer_names, geo_ratio, res_names,
        zoom=min(80, len(res_names)),
        n_samples=4,
        save_path=sp("rq2_bias_sampled"),
        figsize=FIG_SIZES["rq2_sampled"],
        suptitle=f"Geometric attention across model depth  —  {label}",
    )

    # ── Summary text file ─────────────────────────────────────────────
    num_heads   = geo_raw.shape[1]
    valid_corrs = corr_mat[~np.isnan(corr_mat)]
    n_geo_heads = int((geo_ratio > GEO_THRESHOLD).sum())

    # Best correlated head (highest Spearman r)
    n_total_heads   = len(layer_names) * num_heads
    SEQ_R_THRESHOLD = 0.5   # |r| below this → head is "sequential" not structural
    if valid_corrs.size > 0:
        best_flat     = int(np.nanargmax(corr_mat))
        best_layer    = layer_labels[best_flat // num_heads]
        best_head     = best_flat % num_heads
        best_r        = float(np.nanmax(corr_mat))
        corr_mean     = float(valid_corrs.mean())
        corr_std      = float(valid_corrs.std())
        corr_mean_abs = float(np.abs(valid_corrs).mean())
        n_sequential  = int((np.abs(valid_corrs) < SEQ_R_THRESHOLD).sum())
    else:
        best_layer = best_head = best_r = corr_mean = corr_std = corr_mean_abs = float("nan")
        n_sequential = 0

    # Diagonal (sequential) correlation stats — all heads
    valid_diag_all = diag_corr_mat_all[~np.isnan(diag_corr_mat_all)]
    if valid_diag_all.size > 0:
        diag_all_mean     = float(valid_diag_all.mean())
        diag_all_mean_abs = float(np.abs(valid_diag_all).mean())
        n_diag_strong_all = int((np.abs(valid_diag_all) >= SEQ_R_THRESHOLD).sum())
    else:
        diag_all_mean = diag_all_mean_abs = float("nan")
        n_diag_strong_all = 0

    # Diagonal (sequential) correlation stats — geo heads only
    valid_diag = diag_corr_mat[~np.isnan(diag_corr_mat)]
    if valid_diag.size > 0:
        diag_mean     = float(valid_diag.mean())
        diag_mean_abs = float(np.abs(valid_diag).mean())
        n_diag_strong = int((np.abs(valid_diag) >= SEQ_R_THRESHOLD).sum())
    else:
        diag_mean = diag_mean_abs = float("nan")
        n_diag_strong = 0

    # Seq-window mass stats (window=15)
    valid_mass15 = seq_mass[15][~np.isnan(seq_mass[15])]
    MASS_THRESHOLD = 0.5
    if valid_mass15.size > 0:
        mass15_mean  = float(valid_mass15.mean())
        mass15_max   = float(valid_mass15.max())
        n_mass_high  = int((valid_mass15 >= MASS_THRESHOLD).sum())
    else:
        mass15_mean = mass15_max = float("nan")
        n_mass_high = 0

    # Toeplitz stats — all heads
    valid_toep = toeplitz_mat[~np.isnan(toeplitz_mat)]
    TOEP_THRESHOLD = 0.8
    if valid_toep.size > 0:
        toep_mean    = float(valid_toep.mean())
        toep_max     = float(valid_toep.max())
        n_toep_high  = int((valid_toep >= TOEP_THRESHOLD).sum())
    else:
        toep_mean = toep_max = float("nan")
        n_toep_high = 0

    summary_lines = [
        f"Structure summary: {label} ({example})",
        f"{'='*50}",
        f"Residues / tokens : {len(res_names)}",
        f"Pairformer layers  : {len(layer_names)}",
        f"Attention heads    : {num_heads}",
        f"",
        f"── KL divergence (raw) ──────────────────────────",
        f"  Mean geo KL        : {geo_raw.mean():.4f}",
        f"  Mean sem KL        : {sem_raw.mean():.4f}",
        f"  Geo / (geo+sem)    : {geo_ratio.mean():.4f}",
        f"  Sem / (geo+sem)    : {sem_ratio.mean():.4f}",
        f"",
        f"── Raw KL (95th-pct normalised) ─────────────────",
        f"  Mean geo (norm.)   : {geo_norm.mean():.4f}",
        f"  Mean sem (norm.)   : {sem_norm.mean():.4f}",
        f"",
        f"── Bias–structure Spearman r (geo threshold {GEO_THRESHOLD:.0%}) ──",
        f"  Total heads        : {n_total_heads}  ({len(layer_names)} layers × {num_heads} heads)",
        f"  Geo heads (ratio ≥ {GEO_THRESHOLD:.0%})  : {n_geo_heads} / {n_total_heads}",
        f"  Sequential geo heads (|r| < {SEQ_R_THRESHOLD})  : {n_sequential} / {n_geo_heads}",
        f"  Mean Spearman r    : {corr_mean:.4f}",
        f"  Mean |Spearman r|  : {corr_mean_abs:.4f}",
        f"  Std  Spearman r    : {corr_std:.4f}",
        f"  Best head          : layer {best_layer}, head {best_head}  (r = {best_r:.4f})",
        f"",
        f"── Bias–diagonal Spearman r (all heads) ────────",
        f"  Mean Spearman r    : {diag_all_mean:.4f}",
        f"  Mean |Spearman r|  : {diag_all_mean_abs:.4f}",
        f"  Diagonal heads (|r| ≥ {SEQ_R_THRESHOLD})  : {n_diag_strong_all} / {n_total_heads}",
        f"",
        f"── Bias–diagonal Spearman r (geo threshold {GEO_THRESHOLD:.0%}) ──",
        f"  Mean Spearman r    : {diag_mean:.4f}",
        f"  Mean |Spearman r|  : {diag_mean_abs:.4f}",
        f"  Diagonal geo heads (|r| ≥ {SEQ_R_THRESHOLD})  : {n_diag_strong} / {n_geo_heads}",
        f"",
        f"── Toeplitz η² (all heads) ──────────────────────",
        f"  Mean η²            : {toep_mean:.4f}",
        f"  Max  η²            : {toep_max:.4f}",
        f"  High-Toeplitz heads (η² ≥ {TOEP_THRESHOLD})  : {n_toep_high} / {n_total_heads}",
    ]

    summary_path = sp("summary")
    # Strip .png added by sp() — we want .txt
    summary_path = summary_path.replace(".png", ".txt")
    with open(summary_path, "w") as f:
        f.write("\n".join(summary_lines) + "\n")
    print(f"  Summary → {summary_path}")

    print(f"  All figures saved to {out_dir}")
    return sem_line_ratio, sem_line_raw, geo_line_ratio, geo_line_raw


# ======================================================================
# Main loop — function boundary guarantees full cleanup
# ======================================================================
sem_lines_ratio = {}
geo_lines_ratio = {}
sem_lines_raw   = {}
geo_lines_raw   = {}

for example in EXAMPLES:
    result = process_example(example)

    # Close every figure the plot functions left open
    plt.close("all")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    if result is None:
        continue
    sem_lines_ratio[example] = result[0]
    sem_lines_raw[example]   = result[1]
    geo_lines_ratio[example] = result[2]
    geo_lines_raw[example]   = result[3]

# ======================================================================
# Multi-protein overlay
# ======================================================================
print(f"\nGenerating multi-protein overlay …")

fs = FIG_SIZES["overlay"]
fig, axes = plt.subplots(2, 2, figsize=fs,
                         gridspec_kw={"hspace": 0.45, "wspace": 0.35})

overlay_configs = [
    (axes[0, 0], sem_lines_ratio, "Semantic score (ratio)"),
    (axes[0, 1], geo_lines_ratio, "Geometric score (ratio)"),
    (axes[1, 0], sem_lines_raw,   "Semantic score (raw norm.)"),
    (axes[1, 1], geo_lines_raw,   "Geometric score (raw norm.)"),
]

for ax, lines, title in overlay_configs:
    for name, line in lines.items():
        ax.plot(line, color=EXAMPLE_COLORS.get(name, "gray"), lw=1.8, label=EXAMPLE_LABELS[name])
    ax.set_xlabel("Layer")
    ax.set_ylabel("Mean score")
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)
    ax.set_ylim(bottom=0)

fig.suptitle("Geo / Sem routing across proteins", fontsize=13, weight="bold")
plt.tight_layout()
fig.savefig(f"{GLOBAL_DIR}/multi_protein_overlay.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"  Overlay saved to {GLOBAL_DIR}/multi_protein_overlay.png")

print("\nPipeline complete.")

# Scale experiment — test set

Runs inference over ~50 diverse single-chain proteins and computes metrics on-the-fly.  
Full activations are **never saved** — only compact per-protein `.npz` files (~KB each).  
Estimated runtime: ~1–3 min per protein on a T4/A100.

In [ ]:
import os, time, urllib.request, textwrap
import numpy as np

# ── Output directory ──────────────────────────────────────────────────────────
DRIVE         = "/content/drive/MyDrive"
SCALE_DIR     = f"{DRIVE}/scale_experiments"
SCALE_RAW_DIR = f"{SCALE_DIR}/metrics"   # compact .npz per protein
SCALE_YAML_DIR = f"{SCALE_DIR}/yamls"    # temporary YAML files
os.makedirs(SCALE_RAW_DIR,  exist_ok=True)
os.makedirs(SCALE_YAML_DIR, exist_ok=True)

# ── UniProt fetch helper ──────────────────────────────────────────────────────
def fetch_uniprot_sequence(accession: str, timeout: int = 30) -> str | None:
    """Fetch canonical sequence from UniProt REST API.  Returns None on failure."""
    url = f"https://rest.uniprot.org/uniprotkb/{accession}.fasta"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            lines = r.read().decode().strip().split("\n")
        return "".join(l for l in lines if not l.startswith(">"))
    except Exception as e:
        print(f"    [fetch error] {accession}: {e}")
        return None

# ── Curated test set ──────────────────────────────────────────────────────────
# (name, uniprot_id, fold_class, approx_length)
# Covers diverse lengths (40–400 aa) and fold topologies.
# Sequences are fetched at runtime so any stale IDs fail gracefully.
SCALE_TEST_SET = [
    # ── tiny / small (< 100 aa) ───────────────────────────────────────────
    ("ubiquitin",          "P0CG48",  "beta-grasp",   76),
    ("acyl_carrier_prot",  "P0A6A8",  "4-helix",      77),
    ("barstar",            "P11540",  "alpha+beta",   89),
    ("thioredoxin_ecoli",  "P0AA25",  "TRX-fold",     109),
    ("cold_shock_cspb",    "P32081",  "OB-fold",      67),

    # ── medium (100–170 aa) ───────────────────────────────────────────────
    ("barnase",            "P00648",  "alpha+beta",   110),
    ("thioredoxin_human",  "P10599",  "TRX-fold",     105),
    ("rnase_a",            "P61823",  "alpha+beta",   124),
    ("lysozyme_hen",       "P00698",  "alpha+beta",   147),
    ("lysozyme_human",     "P61626",  "alpha+beta",   148),
    ("hemoglobin_alpha",   "P69905",  "all-alpha",    142),
    ("hemoglobin_beta",    "P68871",  "all-alpha",    147),
    ("alpha_lactalbumin",  "P00709",  "alpha+beta",   142),
    ("dhfr_ecoli",         "P0ABQ4",  "alpha+beta",   159),
    ("cyclophilin_a",      "P62937",  "beta-barrel",  165),
    ("myoglobin_whale",    "P02185",  "all-alpha",    154),
    ("myoglobin_horse",    "P02144",  "all-alpha",    154),
    ("calmodulin",         "P0DP23",  "all-alpha",    149),
    ("staphyl_nuclease",   "P00644",  "OB+beta",      149),
    ("rubredoxin",         "P0A3E0",  "coil/beta",    54),
    ("protein_l",          "Q53533",  "beta-grasp",   164),
    ("hiv_protease",       "P03366",  "beta-barrel",  99),
    ("triosephosphate_iso","P0A858",  "TIM-barrel",   248),
    ("flavodoxin_ecoli",   "P0ABN9",  "FMN-binding",  176),

    # ── longer (170–400 aa) ───────────────────────────────────────────────
    ("gfp",                "P42212",  "beta-barrel",  238),
    ("adenylate_kinase",   "P69441",  "alpha+beta",   214),
    ("pcna_human",         "P12004",  "beta-clamp",   261),
    ("glutaredoxin_2",     "O14828",  "TRX-fold",     164),
    ("carbonic_anhydrase", "P00915",  "beta-helix",   261),
    ("dihydro_orotase",    "P0A7E1",  "TIM-barrel",   348),
    ("trypsin",            "P00760",  "beta-barrel",  220),
    ("elastase",           "P00772",  "beta-barrel",  240),
    ("subtilisin",         "P00780",  "alpha+beta",   381),
    ("adk_from_yeast",     "P07170",  "alpha+beta",   220),
    ("superoxide_dis_mn",  "P00448",  "all-alpha",    207),
    ("cytochrome_p450cam", "P00183",  "all-alpha",    414),

    # ── additional diversity ──────────────────────────────────────────────
    ("groel_apical",       "P0A6F5",  "alpha+beta",   548),  # large, may be slow
    ("ras_protein",        "P01112",  "alpha+beta",   189),
    ("calb2_calbindin",    "P05937",  "EF-hand",      79),
    ("hsp70_nbd",          "P0A6Y8",  "alpha+beta",   554),  # large
    ("interleukin_2",      "P60568",  "4-helix",      153),
    ("ribosomal_l12",      "P0A7K2",  "alpha+beta",   121),
    ("copper_azurin",      "P00121",  "greek-key",    128),
    ("plastocyanin",       "P00298",  "greek-key",    99),
    ("ubiquitin_like_smn", "P06659",  "TGS-fold",     168),
    ("calmodulin_2",       "P0DP24",  "EF-hand",      149),  # second isoform
    ("ribonuclease_h",     "P0A7Y4",  "alpha+beta",   155),
    ("gyrase_b_atpase",    "P0AES4",  "GHKL-ATPase",  220),
    ("cheY",               "P0AE67",  "TIM-barrel",   129),
    ("src_sh3",            "P12931",  "SH3",          57),
]

print(f"Test set: {len(SCALE_TEST_SET)} proteins")
print(f"Length range: {min(t[3] for t in SCALE_TEST_SET)}–{max(t[3] for t in SCALE_TEST_SET)} aa (approx)")


# Scale experiment — inference loop

For each protein: fetch sequence → write YAML → run Boltz → compute metrics → save `.npz` → free memory.  
Proteins already processed are skipped automatically (safe to restart).

In [ ]:
import gc, re, textwrap, time
import torch
import boltz
from boltz.analysis.interp import (
    register_metadata_hook,
    decode_res_types,
    get_recycling_step,
    compute_scale_metrics,
)

# ── Settings ──────────────────────────────────────────────────────────────────
MAX_SEQ_LEN   = 400   # skip proteins longer than this (memory)
RECYCLING_STEPS = 1   # 1 is enough to capture head specialisation; 3 for production
KL_TRIALS       = 3   # 3 is fast; 5 is slightly more stable
DIFFUSION_SAMPLES = 1

# ── Load model once ───────────────────────────────────────────────────────────
model = boltz.load_model("boltz1", device="cuda", use_kernels=False)

def _layer_idx(name):
    m = re.search(r"layers\.(\d+)", name)
    return int(m.group(1)) if m else -1

# ── Inference + metric computation per protein ────────────────────────────────
succeeded, skipped, failed = [], [], []
t_start = time.time()

for (prot_name, uniprot_id, fold_class, approx_len) in SCALE_TEST_SET:

    save_path = f"{SCALE_RAW_DIR}/{prot_name}.npz"
    if os.path.exists(save_path):
        print(f"[skip]  {prot_name}  (already computed)")
        skipped.append(prot_name)
        continue

    if approx_len > MAX_SEQ_LEN:
        print(f"[skip]  {prot_name}  (approx len {approx_len} > {MAX_SEQ_LEN})")
        skipped.append(prot_name)
        continue

    print(f"\n{'─'*55}")
    print(f"[{prot_name}]  UniProt={uniprot_id}  fold={fold_class}")

    # 1. Fetch sequence
    seq = fetch_uniprot_sequence(uniprot_id)
    if seq is None or len(seq) < 20:
        print(f"  [FAIL] Could not fetch sequence")
        failed.append(prot_name)
        continue

    actual_len = len(seq)
    if actual_len > MAX_SEQ_LEN:
        print(f"  [skip]  actual length {actual_len} > {MAX_SEQ_LEN}")
        skipped.append(prot_name)
        continue

    print(f"  Sequence length: {actual_len} aa")

    # 2. Write a temporary YAML
    yaml_path = f"{SCALE_YAML_DIR}/{prot_name}.yaml"
    with open(yaml_path, "w") as f:
        f.write(textwrap.dedent(f"""\
            version: 1
            sequences:
              - protein:
                  id: A
                  sequence: {seq}
            """))

    # 3. Register hooks
    metadata, meta_handle = register_metadata_hook(model)
    activations = {}

    def _make_hook(layer_name, comp_name):
        def hook(module, inp, out):
            # Overwrite on each recycling step → keeps only final step
            activations.setdefault(layer_name, {})[comp_name] = \
                out.detach().cpu().clone()
        return hook

    hook_handles = []
    for mod_name, module in model.named_modules():
        if "AttentionPairBias" in type(module).__name__:
            hook_handles += [
                module.proj_q.register_forward_hook(_make_hook(mod_name, "q")),
                module.proj_k.register_forward_hook(_make_hook(mod_name, "k")),
                module.proj_z.register_forward_hook(_make_hook(mod_name, "bias")),
            ]

    # 4. Run inference
    try:
        t0 = time.time()
        results = boltz.predict(
            model, yaml_path,
            use_msa_server=False,         # single-sequence mode for speed
            recycling_steps=RECYCLING_STEPS,
            diffusion_samples=DIFFUSION_SAMPLES,
        )
        coords = results[0]["coords"]     # [samples, atoms, 3]
        t_inf = time.time() - t0
        print(f"  Inference: {t_inf:.1f} s")
    except Exception as e:
        print(f"  [FAIL] Inference error: {e}")
        for h in hook_handles: h.remove()
        meta_handle.remove()
        activations.clear()
        failed.append(prot_name)
        continue

    # 5. Decode residues
    res_names = decode_res_types(
        metadata["res_type"], metadata.get("token_pad_mask")
    )

    # 6. Compute compact metrics (no activations saved to disk)
    layer_names = sorted(
        [k for k in activations if k.startswith("pairformer_module.")],
        key=_layer_idx,
    )

    # The hook pattern here overwrites each recycling step, so activations
    # already hold only the final step — pass through get_recycling_step
    # to normalise into the expected {layer: {comp: tensor}} format.
    acts_single = {
        layer: {comp: [tensor] for comp, tensor in comps.items()}
        for layer, comps in activations.items()
    }
    acts_final = get_recycling_step(acts_single, step=-1)

    try:
        metrics = compute_scale_metrics(
            acts_final, layer_names, coords, res_names,
            num_kl_trials=KL_TRIALS,
        )
    except Exception as e:
        print(f"  [FAIL] Metric computation error: {e}")
        for h in hook_handles: h.remove()
        meta_handle.remove()
        activations.clear()
        del acts_single, acts_final
        failed.append(prot_name)
        continue

    # 7. Save compact metrics + metadata
    np.savez_compressed(
        save_path,
        uniprot_id=np.array(uniprot_id),
        fold_class=np.array(fold_class),
        **metrics,
    )
    print(f"  Saved → {save_path}")
    succeeded.append(prot_name)

    # 8. Free all tensors
    for h in hook_handles: h.remove()
    meta_handle.remove()
    activations.clear()
    del acts_single, acts_final, coords, metrics, results
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

# ── Summary ───────────────────────────────────────────────────────────────────
t_total = time.time() - t_start
print(f"\n{'='*55}")
print(f"Scale loop complete in {t_total/60:.1f} min")
print(f"  Succeeded : {len(succeeded)}")
print(f"  Skipped   : {len(skipped)}")
print(f"  Failed    : {len(failed)}")
if failed:
    print(f"  Failed names: {failed}")
print(f"{'='*55}")


# Scale experiment — analysis and figures

Aggregates all per-protein `.npz` files and produces four publication-quality figures:
1. Mean geo / sem ratio per layer (±std)  
2. Bias–structure Spearman r per layer (±std)  
3. Per-protein scatter vs sequence length  
4. Aggregate layer × head heatmaps

In [ ]:
import os, gc
import numpy as np
import matplotlib.pyplot as plt
from boltz.analysis.interp import aggregate_scale_results, plot_scale_summary

DRIVE         = "/content/drive/MyDrive"
SCALE_DIR     = f"{DRIVE}/scale_experiments"
SCALE_RAW_DIR = f"{SCALE_DIR}/metrics"
SCALE_FIG_DIR = f"{SCALE_DIR}/figures"
os.makedirs(SCALE_FIG_DIR, exist_ok=True)

# ── Load and aggregate all per-protein metrics ───────────────────────────────
print("Loading per-protein metrics …")
agg = aggregate_scale_results(SCALE_RAW_DIR)
P, L, H = agg["geo_ratio"].shape

print(f"  Loaded {P} proteins")
print(f"  Layers: {L}   Heads per layer: {H}")
print(f"  Sequence lengths: {agg['n_residues'].min()}–{agg['n_residues'].max()}")

# ── Produce summary figures ───────────────────────────────────────────────────
plot_scale_summary(
    agg,
    geo_threshold=0.5,
    struct_threshold=0.3,
    save_dir=SCALE_FIG_DIR,
)

# ── Per-fold-class breakdown (geo ratio) ─────────────────────────────────────
# Load fold labels from the saved npz files
fold_labels = []
for name in agg["proteins"]:
    p = f"{SCALE_RAW_DIR}/{name}.npz"
    d = np.load(p, allow_pickle=True)
    fold_labels.append(str(d["fold_class"]))

unique_folds = sorted(set(fold_labels))
fold_geo_means = {f: [] for f in unique_folds}
for i, fold in enumerate(fold_labels):
    fold_geo_means[fold].append(float(agg["geo_ratio"][i].mean()))

fig, ax = plt.subplots(figsize=(10, 4))
positions = range(len(unique_folds))
means = [np.mean(fold_geo_means[f]) for f in unique_folds]
stds  = [np.std(fold_geo_means[f])  for f in unique_folds]
ax.bar(positions, means, yerr=stds, capsize=4, color="#E07B7B", alpha=0.8)
ax.set_xticks(positions)
ax.set_xticklabels(unique_folds, rotation=40, ha="right", fontsize=9)
ax.set_ylabel("Mean geo ratio (averaged over layers & heads)")
ax.set_title(f"Geometric routing by fold class  ({P} proteins)")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.2, axis="y")
ax.axhline(0.5, color="gray", lw=0.8, ls="--", label="geo = sem")
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(f"{SCALE_FIG_DIR}/scale_by_fold_class.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"\nAll scale figures saved to {SCALE_FIG_DIR}")
gc.collect()
